In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import spikeinterface as si
import spikeinterface.preprocessing as spre
from probeinterface.plotting import plot_probe as prb_plotting

from h5d_probe_and_ks4_prep import load_probe_from_chanmap, write_ks4_chanmap

In [ ]:
''' File paths '''
root_dir = "C:/Users/Isabel/Documents/data_temp/"

# session params
bird_id = "SLV132"
session_id = "250310"
ephys_id = "SLV132_250310_122522"
map_file = 'H5D_128Chan_probeMap_seperateFB.mat'

# path to .rhd intan file
intan_folder = f"{root_dir}{bird_id}_{session_id}/{ephys_id}/"

# path to various info files
map_file_path = f'Z:/Isabel/ephys/channel_maps/{map_file}'

In [ ]:
''' Add the probe map and check channel layout '''
probe, device_idx, shank_idx = load_probe_from_chanmap(map_file_path) # todo

# define probe properties
n_channels = device_idx.shape[0]
probe.set_device_channel_indices(np.arange(n_channels))  # positional match into recording
recording = recording.set_probe(probe)
recording.set_property("channel_name", [f"e{c}" for c in device_idx])
recording.set_property("group", shank_idx)

# check for noisy channels
noise_level = si.get_noise_levels(spre.common_reference(spre.highpass_filter(recording)))
plt.plot(noise_level)

In [ ]:
# set noise threshold and remove noisy channels
noise_thresh = 16
keep_ch_idx = noise_level < noise_thresh
bad_device_idx = device_idx[~keep_ch_idx]          # hardware channel #s to exclude
recording_good = recording.select_channels(recording.channel_ids[keep_ch_idx])

print(recording_good.get_num_channels())
print(recording_good.get_property("group").shape)
print(recording_good.get_property("channel_name").shape)
print(recording_good.get_channel_locations().shape)

fig, ax = plt.subplots(1, 1, figsize=(8, 8))
prb_plotting(recording_good.get_probe(), ax=ax)